# Topic 2: Inflation and Macroeconomic Data Download
## DFA vs. LSTM for Inflation Prediction

This notebook downloads U.S. macroeconomic data for inflation analysis.

**Data Requirements:**
- ≥150 observations (monthly data from 1990-2024 = 400+ observations)
- ≥2 variables (multiple macroeconomic indicators)

**Note:** Requires a free FRED API key:
1. Sign up at: https://fred.stlouisfed.org/
2. Get API key: https://fred.stlouisfed.org/docs/api/api_key.html
3. Install: `pip install fredapi`


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


## Download Function


In [ ]:
def download_inflation_data_fred(api_key=None):
    """
    Download U.S. macroeconomic indicators from FRED.
    Requires FRED API key (free at https://fred.stlouisfed.org/docs/api/api_key.html)
    """
    try:
        from fredapi import Fred
    except ImportError:
        print("Installing fredapi: pip install fredapi")
        print("Get API key: https://fred.stlouisfed.org/docs/api/api_key.html")
        return None
    
    if api_key is None:
        print("FRED API key required. Get one at: https://fred.stlouisfed.org/docs/api/api_key.html")
        return None
    
    print("Downloading macroeconomic data from FRED...")
    fred = Fred(api_key=api_key)
    
    # Define FRED series IDs for key macroeconomic indicators
    series_info = {
        'CPI': {'id': 'CPIAUCSL', 'name': 'Consumer Price Index'},
        'Core_CPI': {'id': 'CPILFESL', 'name': 'Core CPI'},
        'Unemployment': {'id': 'UNRATE', 'name': 'Unemployment Rate'},
        'Fed_Funds_Rate': {'id': 'FEDFUNDS', 'name': 'Federal Funds Rate'},
        'GDP': {'id': 'GDPC1', 'name': 'Real GDP'},
        'Money_Supply_M2': {'id': 'M2SL', 'name': 'M2 Money Stock'},
        'Oil_Price': {'id': 'DCOILWTICO', 'name': 'Oil Prices'},
        'Industrial_Production': {'id': 'INDPRO', 'name': 'Industrial Production'},
    }
    
    data_dict = {}
    start_date = '1990-01-01'
    
    print(f"\nDownloading data from {start_date} to present...")
    
    for var_name, info in series_info.items():
        try:
            print(f"Downloading {var_name} ({info['id']})...")
            series = fred.get_series(info['id'], start=start_date)
            if not series.empty:
                data_dict[var_name] = series
                print(f"  ✓ {len(series)} observations")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    if data_dict:
        data = pd.DataFrame(data_dict)
        # Calculate inflation rate (YoY % change)
        if 'CPI' in data.columns:
            data['Inflation_Rate'] = data['CPI'].pct_change(12) * 100
        data = data.dropna()
        print(f"\n✓ Successfully downloaded {len(data)} observations")
        return data
    return None


## Download Data

**Set your FRED API key below and run the cell:**


In [ ]:
# Set your FRED API key here
FRED_API_KEY = 'YOUR_API_KEY_HERE'  # Replace with your actual API key

# Download data
data = download_inflation_data_fred(api_key=FRED_API_KEY)


## Save and Display Data


In [ ]:
if data is not None and len(data) >= 150:
    # Save to CSV
    filename = 'inflation_data.csv'
    data.to_csv(filename)
    print(f"\n✓ Data saved to {filename}")
    
    # Display summary
    print("\n" + "=" * 60)
    print("DATA SUMMARY")
    print("=" * 60)
    print(data.describe())
    print("\nFirst few rows:")
    print(data.head())
    print("\nLast few rows:")
    print(data.tail())
    
    print("\nMissing values:")
    print(data.isnull().sum())
    
    print(f"\nData frequency: {pd.infer_freq(data.index)}")
    print(f"Total observations: {len(data)}")
else:
    print("\n✗ Data download failed or insufficient data.")
    print("  Make sure you set your FRED API key correctly.")
